# Taller Semana 7: Funciones, Modularización y Refactorización
**Curso:** Lógica Computacional · **Programa:** Ingeniería de Datos e IA · **Semana:** 7 de 14
**Estudiante:** Santiago Suarez Lopez
**Dataset de Trabajo:** Mortalidad por accidentes de tránsito Bogotá (`osb_evento_transporte.csv`)

> Antes de empezar (en tu terminal / repo):
> ```bash
> git status
> git pull
> git add .
> git commit -m "chore: establece linea base antes de refactorizar"
> ```

## 0. Carga del dataset (funciona en Colab y en VS Code / Jupyter local)

In [20]:
import os
import csv

NOMBRE_CSV = "osb_evento_transporte.csv"
if not os.path.exists(NOMBRE_CSV):
    try:
        from google.colab import files
        print("Selecciona el archivo osb_evento_transporte.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError(
            "No se encontró osb_evento_transporte.csv. Ponlo en la misma carpeta del notebook."
        )

with open(NOMBRE_CSV, encoding="utf-8-sig") as f:
    lector = csv.reader(f, delimiter=";")
    encabezado = next(lector)
    mi_dataset_lista = [fila for fila in lector]

columna_ano = encabezado.index("ANO")
columna_casos = encabezado.index("casos")
columna_sexo = encabezado.index("Sexo")
columna_condicion = encabezado.index("CONDICION_DE_LA_VICTIMA_AT_")
columna_localidad = encabezado.index("LOCALIDAD")
columna_ciclo = encabezado.index("CICLO_VITAL")

print("Filas cargadas:", len(mi_dataset_lista))
print("Columnas:", len(encabezado))

Filas cargadas: 13138
Columnas: 18


## Línea base (ANTES de refactorizar)
Resultados de la Semana 6 (tabla de casos de prueba, caso 6), con las filas `Bogotá` excluidas:
Total = 6569, Hombres = 5155, Mujeres = 1414. Estos son los valores que **no deben cambiar**.

In [21]:
LINEA_BASE = {"total": 6569, "hombres": 5155, "mujeres": 1414}
LINEA_BASE

{'total': 6569, 'hombres': 5155, 'mujeres': 1414}

---
# Parte 1. Retos 1 a 8 (funciones básicas)

## Reto 1: Primera función

In [22]:
def mostrar_mensaje():
    print("Procesando Dataset de Trabajo")

mostrar_mensaje()

Procesando Dataset de Trabajo


1. La **definición** es `def mostrar_mensaje(): ...`.
2. La **invocación** es la línea `mostrar_mensaje()`.
3. Su **responsabilidad** es mostrar el mensaje de inicio del procesamiento.

## Reto 2: Parámetro y argumento

In [23]:
def saludar(nombre):
    print(f"Hola, {nombre}")

saludar("Ana")
saludar("Carlos")
saludar("María")

Hola, Ana
Hola, Carlos
Hola, María


1. **Parámetro:** `nombre` (lo que va entre paréntesis en la definición).
2. **Argumento:** `"Ana"`, `"Carlos"`, `"María"` (el valor real al llamar la función).
3. **Ventaja:** la misma función sirve para muchos valores sin copiar código.

## Reto 3: Función con retorno

In [24]:
def duplicar(valor):
    return valor * 2

print(duplicar(5))
print(duplicar(0))
print(duplicar(10))

10
0
20


1. Recibe un número.  2. Calcula su doble.  3. Retorna ese doble.
4. El resultado **no queda guardado** en la función: queda donde se llamó (se puede guardar en una variable, por ejemplo `r = duplicar(5)`, o usarse directo).

## Reto 4: `print()` vs `return`

In [25]:
def mostrar_resultado(valor):
    print(valor)

def obtener_resultado(valor):
    return valor

a = mostrar_resultado(10)
b = obtener_resultado(10)

print("a =", a)
print("b =", b)
print("b + 5 =", b + 5)

10
a = None
b = 10
b + 5 = 15


**¿Cuál permite que otro bloque use el resultado?** `obtener_resultado()`, porque con `return` el valor sale de la función y se puede guardar o usar en otros cálculos. `print()` solo lo muestra en pantalla y la función devuelve `None`.

## Reto 5: Función de clasificación
Mi dataset es categórico, así que mi `clasificar_valor()` conserva la regla de la Semana 6: según la condición de la víctima retorna **Conductor, Peatón, Pasajero o Sin información** (en lugar de ALTO/NORMAL/BAJO).

In [26]:
CATEGORIAS_CONOCIDAS = ("Conductor", "Peatón", "Pasajero")

def clasificar_valor(condicion):
    if condicion in CATEGORIAS_CONOCIDAS:
        return condicion
    return "Sin información"

print(clasificar_valor("Conductor"))
print(clasificar_valor("Peatón"))
print(clasificar_valor("Pasajero"))
print(clasificar_valor(""))

Conductor
Peatón
Pasajero
Sin información


**¿Por qué reutilizar `clasificar_valor()`?** Porque la regla vive en un solo lugar: si cambia, se corrige una vez y todo el programa se actualiza, sin riesgo de que dos copias den resultados distintos.

## Reto 6: Contador mediante una función

In [27]:
def contar_registros(valores):
    contador = 0
    for _ in valores:
        contador += 1
    return contador

print(contar_registros([22, 55, 72, 38, 61]))   # 5

5


**¿Por qué iniciar el contador dentro?** Para que cada llamada empiece desde 0. Si estuviera afuera (global), la segunda llamada seguiría sumando sobre el resultado anterior.

## Reto 7: Acumulador mediante una función

In [28]:
def calcular_suma(valores):
    suma = 0
    for valor in valores:
        suma += valor
    return suma

print(calcular_suma([10, 20, 30, 40]))

100


## Reto 8: Clasificación y resumen

In [29]:
def resumir_clasificacion(condiciones):
    conductores = 0
    peatones = 0
    pasajeros = 0
    sin_info = 0
    for c in condiciones:
        categoria = clasificar_valor(c)
        if categoria == "Conductor":
            conductores += 1
        elif categoria == "Peatón":
            peatones += 1
        elif categoria == "Pasajero":
            pasajeros += 1
        else:
            sin_info += 1
    return conductores, peatones, pasajeros, sin_info

muestra = ["Conductor", "Peatón", "Pasajero", "Conductor", ""]
print(resumir_clasificacion(muestra))

(2, 1, 1, 1)


---
# Parte 2. Dataset de Trabajo v0.7 (modularizado)

### Detalle importante que corregí al refactorizar
En la Semana 6 el filtro comparaba contra `"Bogotá"` y `"BOGOTA"` según la celda, pero los datos pueden traer tildes o espacios. Eso hacía que la celda final no excluyera nada (total 13138 en vez de 6569). Ahora `normalizar_texto()` quita tildes, espacios y mayúsculas, y **una sola función** (`es_fila_valida`) decide qué filas se procesan.

In [36]:
import unicodedata
def normalizar_texto(texto):
    texto = texto.strip().upper()
    sin_tildes = unicodedata.normalize("NFD", texto)
    return "".join(c for c in sin_tildes if unicodedata.category(c) != "Mn")

def es_fila_valida(fila):
    localidad = normalizar_texto(fila[columna_localidad])
    return localidad != "BOGOTA" and localidad != ""
def clasificar_valor(condicion):
    if condicion.strip() in CATEGORIAS_CONOCIDAS:
        return condicion.strip()
    return "Sin información"
def procesar_registros(datos, cantidad=None):
    r = {"total": 0, "excluidas": 0, "hombres": 0, "mujeres": 0,
         "conductores": 0, "peatones": 0, "pasajeros": 0, "sin_info": 0,
         "suma_casos": 0.0, "existe_primera_infancia": False}

    if cantidad is not None:
        datos = datos[:cantidad]

    for fila in datos:
        if not es_fila_valida(fila):
            r["excluidas"] += 1
            continue

        r["total"] += 1
        r["suma_casos"] += float(fila[columna_casos])

        sexo = fila[columna_sexo].strip()
        if sexo == "Hombre":
            r["hombres"] += 1
        elif sexo == "Mujer":
            r["mujeres"] += 1

        categoria = clasificar_valor(fila[columna_condicion])
        if categoria == "Conductor":
            r["conductores"] += 1
        elif categoria == "Peatón":
            r["peatones"] += 1
        elif categoria == "Pasajero":
            r["pasajeros"] += 1
        else:
            r["sin_info"] += 1

        if fila[columna_ciclo].startswith("(00 a 05)"):
            r["existe_primera_infancia"] = True

    return r
def mostrar_resumen(total, excluidas, hombres, mujeres, conductores,
                    peatones, pasajeros, sin_info, suma_casos, existe_primera_infancia):
    print(f"Total de registros (casos únicos): {total}")
    print(f"Filas excluidas (Bogotá / vacías): {excluidas}")
    print(f"Hombres: {hombres} | Mujeres: {mujeres}")
    print(f"Conductores: {conductores} | Peatones: {peatones} | Pasajeros: {pasajeros} | Sin información: {sin_info}")
    print(f"Acumulado de casos: {suma_casos}")
    print(f"¿Hay víctimas de Primera Infancia?: {existe_primera_infancia}")

def mostrar_menu():
    """Solo muestra las opciones."""
    print("1. Procesar el dataset completo")
    print("2. Procesar una muestra (primeros N registros)")
    print("3. Ver el último resumen")
    print("0. Salir")

def leer_opcion():
    opcion = input("Elige una opción: ").strip()
    return opcion
def ejecutar_opcion(opcion, datos, ultimo_resultado):
    if opcion == "1":
        ultimo_resultado = procesar_registros(datos)
        mostrar_resumen(**ultimo_resultado)
    elif opcion == "2":
        texto = input("¿Cuántos registros?: ").strip()
        if texto.isdigit() and int(texto) > 0:
            ultimo_resultado = procesar_registros(datos, int(texto))
            mostrar_resumen(**ultimo_resultado)
        else:
            print("Cantidad inválida.")
    elif opcion == "3":
        if ultimo_resultado is None:
            print("Aún no hay resumen. Procesa primero (opción 1 o 2).")
        else:
            mostrar_resumen(**ultimo_resultado)
    else:
        print("Opción inválida.")
    return ultimo_resultado

def main():
    ultimo_resultado = None
    opcion = ""
    while opcion != "0":
        mostrar_menu()
        opcion = leer_opcion()
        if opcion != "0":
            ultimo_resultado = ejecutar_opcion(opcion, mi_dataset_lista, ultimo_resultado)
    print("Programa finalizado.")

## Pruebas DESPUÉS de refactorizar (sin menú)

In [35]:
r = procesar_registros(mi_dataset_lista)
mostrar_resumen(**r)

assert r["hombres"] + r["mujeres"] == r["total"]
assert r["conductores"] + r["peatones"] + r["pasajeros"] + r["sin_info"] == r["total"]
assert r["suma_casos"] == r["total"]

print()
print("Comparación con la línea base de la Semana 6:")
for clave, esperado in LINEA_BASE.items():
    obtenido = r[clave]
    print(f"  {clave}: antes={esperado}  después={obtenido}  ¿coincide? {esperado == obtenido}")

Total de registros (casos únicos): 6569
Filas excluidas (Bogotá / vacías): 6569
Hombres: 5155 | Mujeres: 1414
Conductores: 3135 | Peatones: 2792 | Pasajeros: 597 | Sin información: 45
Acumulado de casos: 6569.0
¿Hay víctimas de Primera Infancia?: True

Comparación con la línea base de la Semana 6:
  total: antes=6569  después=6569  ¿coincide? True
  hombres: antes=5155  después=5155  ¿coincide? True
  mujeres: antes=1414  después=1414  ¿coincide? True


---
# Parte 3. Pruebas, análisis y documentación

## Reto 17: Pruebas antes y después

| Prueba | Antes (Sem. 6) | Después (v0.7) | ¿Coincide? |
|---|---|---|---|
| Clasificación | Conductor/Peatón/Pasajero/Sin info | igual | Sí |
| Procesamiento (total) | 6569 | 6569 | Sí |
| Resumen (Hombres / Mujeres) | 5155 / 1414 | 5155 / 1414 | Sí |
| Opción válida (1) | n/a (no había menú) | muestra el resumen completo | Sí |
| Opción inválida (z) | n/a | "Opción inválida." y vuelve al menú | Sí |
| Salida (0) | n/a | "Programa finalizado." | Sí |

## Retos 10, 12, 13 y 16: respuestas

**Reto 10: `leer_opcion()`** No recibe nada y retorna la opción escrita por el usuario. Usa `return` para que `main()` pueda pasarla a `ejecutar_opcion()`; si solo la imprimiera, el resto del programa no podría usarla.

**Reto 12: `procesar_registros()`** Recibe el dataset (y opcionalmente la cantidad de registros) y retorna un diccionario con los resultados. Contadores: total, excluidas, hombres, mujeres, conductores, peatones, pasajeros, sin información. Acumulador: `suma_casos`. Bandera: `existe_primera_infancia`.

**Reto 13: `mostrar_resumen()`** Solo muestra y no calcula, para poder probar el cálculo sin imprimir y cambiar la presentación sin tocar la lógica. Cada función queda con una sola responsabilidad.

**Reto 16: procedimiento**
1. Ejecuté la versión de la Semana 6 y registré los resultados (6569 / 5155 / 1414).
2. Hice el commit de línea base.
3. Separé el código en funciones, una por una.
4. Volví a ejecutar y comparé: los resultados coinciden.
5. Corregí un error de la versión anterior: el filtro de `Bogotá` no excluía nada por la tilde; ahora lo resuelve `normalizar_texto()`.

## Reto 18: Pruebas de funciones

In [37]:
assert clasificar_valor("Peatón") == "Peatón"
assert clasificar_valor("") == "Sin información"
assert duplicar(5) == 10
assert contar_registros([10, 20, 30]) == 3
assert calcular_suma([10, 20, 30, 40]) == 100
assert normalizar_texto(" Bogotá ") == "BOGOTA"
print("Todas las pruebas de funciones pasaron.")

Todas las pruebas de funciones pasaron.


| Función | Entrada | Esperado | Obtenido |
|---|---|---|---|
| `clasificar_valor` | `"Peatón"` | `"Peatón"` | `"Peatón"` |
| `clasificar_valor` | `""` | `"Sin información"` | `"Sin información"` |
| `contar_registros` | `[10, 20, 30]` | 3 | 3 |
| `calcular_suma` | `[10, 20, 30, 40]` | 100 | 100 |
| `normalizar_texto` | `" Bogotá "` | `"BOGOTA"` | `"BOGOTA"` |

## Reto 15: Variables globales
| Variable | ¿Es global? | ¿Puede ser parámetro? | ¿Puede retornarse? |
|---|---|---|---|
| `mi_dataset_lista` | Sí | Sí (`main` se la pasa a `ejecutar_opcion` y `procesar_registros`) | No |
| `columna_*` (índices) | Sí | Posible, pero son constantes de lectura | No |
| `total`, `cont_hombre`, `acum_casos`... (Sem. 6) | Sí | No | **Sí**: ahora viven dentro de `procesar_registros` y se retornan en un diccionario |
| `CATEGORIAS_CONOCIDAS` | Sí (constante) | Sí | No |

## Reto 19: Análisis de responsabilidad
| Función | ¿Qué recibe? | ¿Qué hace? | ¿Qué retorna? |
|---|---|---|---|
| `mostrar_menu()` | nada | imprime las opciones | nada |
| `leer_opcion()` | nada | pide la opción por teclado | la opción (texto) |
| `clasificar_valor()` | condición de la víctima | decide la categoría | la categoría |
| `procesar_registros()` | dataset y cantidad opcional | recorre, filtra, cuenta, acumula y activa la bandera | diccionario de resultados |
| `mostrar_resumen()` | los resultados sueltos | imprime el resumen | nada |
| `main()` | nada | coordina el ciclo del menú | nada |

Ninguna función hace demasiadas cosas, aunque `procesar_registros()` es la más grande; se podría dividir más adelante (por ejemplo, separar el conteo por sexo).

## Reto 20: Documentación de funciones
- `normalizar_texto(texto)`: recibe un texto y retorna el mismo sin tildes, espacios sobrantes y en mayúsculas.
- `es_fila_valida(fila)`: decide si una fila es un caso único (no es `BOGOTA` ni vacía).
- `clasificar_valor(condicion)`: recibe la condición de la víctima y retorna su categoría.
- `procesar_registros(datos, cantidad)`: procesa los registros y retorna los contadores, acumulador y bandera.
- `mostrar_resumen(...)`: presenta el resumen ya calculado.
- `mostrar_menu()` / `leer_opcion()` / `ejecutar_opcion()`: muestran, leen y coordinan las opciones.
- `main()`: ejecuta el ciclo del programa.

## Reto 14: Estructura final
```text
main()
 ├── mostrar_menu()
 ├── leer_opcion()
 └── ejecutar_opcion()
       ├── procesar_registros()
       │     ├── es_fila_valida() → normalizar_texto()
       │     └── clasificar_valor()
       └── mostrar_resumen()
```
Quedaron fuera de `main()`: mostrar el menú, leer la opción, decidir qué operación ejecutar, procesar, clasificar y mostrar el resumen. `main()` solo coordina el ciclo.

---
# Preguntas de reflexión
1. **¿Qué problema resuelven las funciones?** Evitan repetir código y dividen un problema grande en partes pequeñas y entendibles.
2. **Parámetro vs argumento:** el parámetro es la variable de la definición (`nombre`); el argumento es el valor concreto que se pasa al llamar (`"Ana"`).
3. **¿Por qué `return` es importante?** Porque entrega el resultado al resto del programa para que pueda guardarlo o usarlo.
4. **¿Cuándo conviene una función?** Cuando una tarea se repite o tiene una responsabilidad clara que se puede nombrar.
5. **`print()` vs `return`:** `print()` solo muestra en pantalla; `return` devuelve un valor utilizable.
6. **Ventaja de dividir en funciones:** el código es más claro, se prueba por partes, se reutiliza y es más fácil de mantener.
7. **¿Por qué evitar globales innecesarias?** Cualquier parte del programa puede modificarlas y es difícil rastrear errores; con parámetros y retornos el flujo de datos es explícito.
8. **¿Qué es refactorizar?** Reorganizar el código para mejorar su estructura sin cambiar lo que hace.
9. **Riesgo al refactorizar:** romper algo que ya funcionaba sin darse cuenta.
10. **¿Por qué el commit de línea base?** Permite comparar y volver atrás si la refactorización falla.

---
# Ejecutar el programa con menú
Esta celda va al final porque espera que escribas una opción: `1` (dataset completo), `2` (muestra), `3` (último resumen), `0` (salir). Si usas "Ejecutar todo", todo corre hasta aquí.

In [38]:
main()

1. Procesar el dataset completo
2. Procesar una muestra (primeros N registros)
3. Ver el último resumen
0. Salir
Elige una opción: 1
Total de registros (casos únicos): 6569
Filas excluidas (Bogotá / vacías): 6569
Hombres: 5155 | Mujeres: 1414
Conductores: 3135 | Peatones: 2792 | Pasajeros: 597 | Sin información: 45
Acumulado de casos: 6569.0
¿Hay víctimas de Primera Infancia?: True
1. Procesar el dataset completo
2. Procesar una muestra (primeros N registros)
3. Ver el último resumen
0. Salir
Elige una opción: 0
Programa finalizado.
